In [ ]:
import torch
import timm
import numpy as np
import time
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from torchvision.datasets import ImageNet
from torch.utils.data import DataLoader
from utils import InputHook

In [ ]:
def get_imagenet_batch(path, batch_size, device):
    """Loads a single batch from the ImageNet validation set."""
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    dataset = ImageNet(path, transform=transform, split='val')
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    images, _ = next(iter(loader))
    return images.to(device)

def time_cuda(fn):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    fn()
    torch.cuda.synchronize()
    return time.perf_counter() - t0

In [ ]:
# --- Configuration ---
MODEL_NAME = 'resnet18'

BATCH_SIZES = [8, 16, 32, 64, 128]
IMAGENET_VAL_PATH = '' # Replace with ImageNet Directory
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WARMUP = 3
NUM_TIMED = 5

In [ ]:
model = timm.create_model(MODEL_NAME).to(DEVICE)
model.eval()

batch = get_imagenet_batch(IMAGENET_VAL_PATH, max([8, 16, 32, 64, 128]), DEVICE)
forward_only=[]
forward_with_hook=[]
for batch_size in BATCH_SIZES:
    print(f'Processing Batch Size {batch_size}')
    sub_batch=batch[:batch_size]
    print('...forward only...')
    with torch.no_grad():
        for _ in range(NUM_WARMUP):
            model(sub_batch)

        forward_times = []
        for _ in range(NUM_TIMED):
            t = time_cuda(lambda: model(sub_batch))
            forward_times.append(t)

    forward_only.append(np.mean(forward_times))

    hook = InputHook(model)
    print('...forward with VQKernel...')
    with torch.no_grad():
        for _ in range(NUM_WARMUP):
            hook(sub_batch)

        hook_forward_times = []
        for _ in range(NUM_TIMED):
            t = time_cuda(lambda: hook(sub_batch))
            hook_forward_times.append(t)

    forward_with_hook.append(np.mean(hook_forward_times))

    hook.remove()

In [ ]:
fig,ax=plt.subplots(figsize=(5, 4),dpi=200)
ax.plot(BATCH_SIZES, forward_only, marker='o', label='Fowrad Only', color=plt.cm.winter(0.25))
ax.plot(BATCH_SIZES, forward_with_hook, marker='o', label='Fowrad + VQKernel', color=plt.cm.autumn(0.25))

ax.set_xticks(BATCH_SIZES)
ax.set_xlabel("Batch Size")
ax.set_ylabel("Time (seconds)")
ax.legend(fontsize=8, ncol=2)
ax.grid(linestyle='--',color='grey',alpha=0.25)
plt.tight_layout()
plt.show()